# Read-level bootstrapping with fastQpick

This notebook demonstrates an application of FASTQ sampling **with replacement**, the
capability that distinguishes `fastQpick` from common subsampling tools such as `seqtk
sample`. Sampling reads with replacement turns a single FASTQ file into an arbitrary
number of bootstrap replicates, which lets one attach uncertainty estimates to any
downstream quantity computed from the reads without re-running the wet-lab experiment.

As a self-contained example, a small RNA-seq experiment with known transcript
abundances is simulated. `fastQpick` is then used to draw $B$ bootstrap replicates of
the read file (`fraction=1.0`, `replacement=True`), each replicate is re-quantified, and
the bootstrap distribution of the abundance estimates is compared against the analytic
sampling error. The bootstrap standard errors recover the analytic multinomial standard
errors, confirming that read-level resampling with replacement captures the sampling
uncertainty of the quantification.

In [1]:
import os
import shutil
from collections import Counter

import numpy as np
import pyfastx
import matplotlib.pyplot as plt

from fastQpick import fastQpick

rng = np.random.default_rng(0)

WORKDIR = os.path.abspath("_bootstrap_demo")
FIG_DIR = os.path.abspath("figures")
if os.path.exists(WORKDIR):
    shutil.rmtree(WORKDIR)
os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Simulate an RNA-seq experiment with known abundances

Four transcripts are assigned known relative abundances. $N$ reads are drawn from a
multinomial distribution over the transcripts; each read is written as a transcript-
specific barcode followed by random padding so that the originating transcript can be
recovered exactly from the read sequence. This stands in for the alignment or
pseudo-alignment step of a real quantification pipeline.

In [2]:
TRANSCRIPTS = ["TXA", "TXB", "TXC", "TXD"]
TRUE_PROPS = np.array([0.50, 0.30, 0.15, 0.05])
BARCODES = {"TXA": "AAAACCCC", "TXB": "GGGGTTTT", "TXC": "ACACACAC", "TXD": "TGTGTGTG"}
BARCODE_LEN = 8
N_READS = 50_000
READ_PAD = 42

assignments = rng.choice(len(TRANSCRIPTS), size=N_READS, p=TRUE_PROPS)
bases = np.array(list("ACGT"))

input_fastq = os.path.join(WORKDIR, "reads.fastq")
with open(input_fastq, "w") as fh:
    for i, t in enumerate(assignments):
        barcode = BARCODES[TRANSCRIPTS[t]]
        pad = "".join(rng.choice(bases, size=READ_PAD))
        seq = barcode + pad
        qual = "I" * len(seq)
        fh.write(f"@read{i}\n{seq}\n+\n{qual}\n")

observed_counts = Counter(TRANSCRIPTS[t] for t in assignments)
observed_props = np.array([observed_counts[t] / N_READS for t in TRANSCRIPTS])
print("Wrote", N_READS, "reads to", input_fastq)
for t, p_true, p_obs in zip(TRANSCRIPTS, TRUE_PROPS, observed_props):
    print(f"  {t}: true={p_true:.3f}  observed={p_obs:.4f}")

Wrote 50000 reads to /home/jrich/Desktop/fastQpick/notebooks/_bootstrap_demo/reads.fastq
  TXA: true=0.500  observed=0.4994
  TXB: true=0.300  observed=0.3009
  TXC: true=0.150  observed=0.1493
  TXD: true=0.050  observed=0.0504


## 2. Generate bootstrap replicates with fastQpick

Each bootstrap replicate is a full-size resample of the original FASTQ drawn **with
replacement** (`fraction=1.0`, `replacement=True`). A distinct random seed is used per
replicate, and each replicate is written to its own output directory. Counting the reads
in the input file happens only once: the resulting `fastq_to_length_dict` is reused
across replicates so that the repeated calls only pay for the sampling and writing
steps.

In [3]:
B = 300
length_dict = {input_fastq: N_READS}
rep_dirs = []
for b in range(B):
    out_dir = os.path.join(WORKDIR, f"rep_{b:04d}")
    fastQpick(
        input_files=input_fastq,
        fraction=1.0,
        seed=b + 1,
        output_dir=out_dir,
        replacement=True,
        overwrite=True,
        verbose=False,
        fastq_to_length_dict=length_dict,
    )
    rep_dirs.append(out_dir)
print("Generated", len(rep_dirs), "bootstrap replicates with replacement")

Generated 300 bootstrap replicates with replacement


## 3. Re-quantify each replicate

Each replicate is quantified by recovering the transcript barcode from every read and
computing the proportion of reads assigned to each transcript. This yields a bootstrap
distribution of the abundance estimate for every transcript.

In [4]:
barcode_to_tx = {bc: tx for tx, bc in BARCODES.items()}

def quantify(fastq_path):
    counts = Counter()
    total = 0
    for _, seq, _ in pyfastx.Fastx(fastq_path):
        counts[barcode_to_tx[seq[:BARCODE_LEN]]] += 1
        total += 1
    return np.array([counts[t] / total for t in TRANSCRIPTS])

boot_props = np.vstack([quantify(os.path.join(d, "reads.fastq")) for d in rep_dirs])
print("Bootstrap proportion matrix shape (replicates x transcripts):", boot_props.shape)

Bootstrap proportion matrix shape (replicates x transcripts): (300, 4)


## 4. Bootstrap confidence intervals vs. analytic sampling error

For a proportion estimated from $N$ reads drawn from a multinomial distribution, the
analytic sampling standard error is $\sqrt{p(1-p)/N}$. The read-level bootstrap should
recover this standard error directly from the data, without knowing the underlying
model. The table below reports the observed estimate, the bootstrap standard error, the
percentile 95% confidence interval, and the analytic standard error for each transcript.

In [5]:
boot_se = boot_props.std(axis=0, ddof=1)
ci_lo = np.percentile(boot_props, 2.5, axis=0)
ci_hi = np.percentile(boot_props, 97.5, axis=0)
analytic_se = np.sqrt(observed_props * (1 - observed_props) / N_READS)

print(f"{'tx':>4} {'observed':>9} {'boot_SE':>9} {'analytic_SE':>12} {'95% CI':>22}")
for i, t in enumerate(TRANSCRIPTS):
    ci = f"[{ci_lo[i]:.4f}, {ci_hi[i]:.4f}]"
    print(f"{t:>4} {observed_props[i]:>9.4f} {boot_se[i]:>9.5f} {analytic_se[i]:>12.5f} {ci:>22}")

rel_err = np.abs(boot_se - analytic_se) / analytic_se
print(f"\nMax relative difference between bootstrap and analytic SE: {rel_err.max():.1%}")

  tx  observed   boot_SE  analytic_SE                 95% CI
 TXA    0.4994   0.00228      0.00224       [0.4950, 0.5039]
 TXB    0.3009   0.00204      0.00205       [0.2966, 0.3047]
 TXC    0.1493   0.00165      0.00159       [0.1464, 0.1524]
 TXD    0.0504   0.00096      0.00098       [0.0484, 0.0521]

Max relative difference between bootstrap and analytic SE: 3.4%


In [6]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))

# Panel A: bootstrap distributions with true values and 95% CIs
positions = np.arange(len(TRANSCRIPTS))
parts = ax1.violinplot([boot_props[:, i] for i in range(len(TRANSCRIPTS))],
                       positions=positions, showextrema=False)
for pc in parts['bodies']:
    pc.set_facecolor("#4C72B0")
    pc.set_alpha(0.6)
ax1.vlines(positions, ci_lo, ci_hi, color="#333333", lw=2, label="95% bootstrap CI")
ax1.scatter(positions, TRUE_PROPS, color="#C44E52", zorder=3, label="true proportion")
ax1.set_xticks(positions)
ax1.set_xticklabels(TRANSCRIPTS)
ax1.set_ylabel("estimated proportion")
ax1.set_title("A. Bootstrap distribution of abundance estimates")
ax1.legend(frameon=False, fontsize=9)

# Panel B: bootstrap SE vs analytic SE
lim = max(boot_se.max(), analytic_se.max()) * 1.15
ax2.plot([0, lim], [0, lim], color="#888888", ls="--", lw=1, label="y = x")
ax2.scatter(analytic_se, boot_se, color="#4C72B0", zorder=3)
for i, t in enumerate(TRANSCRIPTS):
    ax2.annotate(t, (analytic_se[i], boot_se[i]), textcoords="offset points", xytext=(6, -2), fontsize=9)
ax2.set_xlim(0, lim)
ax2.set_ylim(0, lim)
ax2.set_xlabel(r"analytic SE  $\sqrt{p(1-p)/N}$")
ax2.set_ylabel("bootstrap SE")
ax2.set_title("B. Bootstrap SE recovers analytic SE")
ax2.legend(frameon=False, fontsize=9)

fig.tight_layout()
out_path = os.path.join(FIG_DIR, "bootstrap_demo.png")
fig.savefig(out_path, dpi=200)
print("Saved figure to", out_path)
plt.show()

Saved figure to /home/jrich/Desktop/fastQpick/notebooks/figures/bootstrap_demo.png


## 5. Summary

Using `fastQpick` to resample reads with replacement, $B=300$ bootstrap replicates were
generated from a single FASTQ file and re-quantified. The bootstrap standard errors of
the transcript abundance estimates match the analytic multinomial standard errors to
within a few percent, and the percentile confidence intervals cover the true
abundances. This is the canonical use case for read-level bootstrapping: propagating
the sampling uncertainty of sequencing into the uncertainty of any downstream estimate.
It is only possible because `fastQpick` supports sampling with replacement, which most
FASTQ subsampling tools do not. The same procedure extends directly to grouped
paired-end files, where the `group_size` option keeps mates synchronized across the
resample.